# IA102 — Notebook 1 — Introduction aux réseaux de neurones

**Durée : 30 minutes** · Objectifs visés : **O1** (bases de l'apprentissage profond) et **O2** (créer des réseaux)

### Plan
1. Le neurone artificiel
2. Les fonctions d'activation
3. Des neurones aux couches, des couches aux réseaux
4. Les tenseurs PyTorch
5. Autograd et rétropropagation
6. Définir un réseau : `nn.Sequential` et `nn.Module`
7. Couche de sortie et fonction de perte

In [ ]:
# Outil de questions de vérification (exécutez cette cellule sans la modifier)
import ipywidgets as widgets
from IPython.display import display, Markdown

def quiz(question, options, bonne, explication):
    """Affiche une question à choix multiples avec rétroaction immédiate."""
    choix = widgets.RadioButtons(options=options, value=None,
                                 layout={"width": "max-content"})
    bouton = widgets.Button(description="Vérifier", button_style="primary")
    sortie = widgets.Output()

    def verifier(_):
        with sortie:
            sortie.clear_output()
            if choix.value is None:
                display(Markdown("Veuillez choisir une réponse."))
            elif choix.value == options[bonne]:
                display(Markdown(f"**Bonne réponse.** {explication}"))
            else:
                display(Markdown("**Réponse incorrecte.** Relisez la section précédente et essayez de nouveau."))

    bouton.on_click(verifier)
    display(Markdown(f"**Question de vérification.** {question}"), choix, bouton, sortie)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
torch.manual_seed(0)

## 1. Le neurone artificiel

Un neurone reçoit un vecteur d'entrées $x = (x_1, \dots, x_d)$, calcule une **combinaison linéaire** pondérée par des **poids** $w$ et décalée par un **biais** $b$, puis applique une **fonction d'activation** $\varphi$ :

$$z = w^\top x + b = \sum_{j=1}^{d} w_j\,x_j + b \qquad\qquad a = \varphi(z)$$

Les poids et le biais sont les **paramètres** du neurone. Sans activation (ou avec l'identité), un neurone est exactement une régression linéaire ; avec la sigmoïde, c'est une régression logistique (IA101, notebook 3).

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.4)); ax.axis("off"); ax.set_xlim(0, 10); ax.set_ylim(0, 4)
entrees, poids = [2.0, -1.0, 0.5], [0.4, 0.3, -0.6]
for i, (x, w) in enumerate(zip(entrees, poids)):
    yy = 3.3 - i * 1.3
    ax.text(0.8, yy, f"x{i+1} = {x}", ha="center", va="center", bbox=dict(boxstyle="round", fc="#eaf1fb", ec="#1f5fa8"))
    ax.annotate("", xy=(4.3, 2.0), xytext=(1.6, yy), arrowprops=dict(arrowstyle="->", color="gray"))
    ax.text(2.7, (yy + 2.0) / 2 + 0.2, f"w{i+1} = {w}", fontsize=9, color="#c0392b")
ax.text(5, 2, "z = Σ wx + b\n(b = 0,1)", ha="center", va="center", fontsize=10, bbox=dict(boxstyle="round,pad=0.5", fc="#fdf2e3", ec="#d68910"))
ax.annotate("", xy=(6.9, 2), xytext=(6.1, 2), arrowprops=dict(arrowstyle="->", color="gray"))
ax.text(7.6, 2, "φ = ReLU", ha="center", va="center", fontsize=10, bbox=dict(boxstyle="round", fc="#e9f5ec", ec="#1e8449"))
ax.annotate("", xy=(9.3, 2), xytext=(8.4, 2), arrowprops=dict(arrowstyle="->", color="gray"))
ax.text(9.6, 2, "a", ha="center", va="center", fontsize=14, fontweight="bold")
ax.set_title("Neurone artificiel : combinaison linéaire, puis activation", fontsize=12); plt.show()

### Exercice 1 — Calculer la sortie d'un neurone

Avec les valeurs du schéma, calculez $z$, puis $a = \text{ReLU}(z) = \max(z, 0)$. Faites d'abord le calcul à la main.

In [ ]:
x = np.array([2.0, -1.0, 0.5])
w = np.array([0.4, 0.3, -0.6])
b = 0.1

z = ...   # À COMPLÉTER
a = ...   # À COMPLÉTER
print(f"z = {z:.2f}, a = {a:.2f}")

<details>
<summary><b>Afficher la solution</b> (après avoir fait l'exercice)</summary>

```python
x = np.array([2.0, -1.0, 0.5])
w = np.array([0.4, 0.3, -0.6])
b = 0.1

z = np.dot(w, x) + b          # 0,8 − 0,3 − 0,3 + 0,1 = 0,3
a = max(z, 0)
print(f"z = {z:.2f}, a = {a:.2f}")
```

</details>

Vérification avec PyTorch : `nn.Linear(3, 1)` implémente exactement $z = w^\top x + b$.

In [ ]:
neurone = nn.Linear(3, 1)
with torch.no_grad():
    neurone.weight[:] = torch.tensor([[0.4, 0.3, -0.6]])
    neurone.bias[:] = torch.tensor([0.1])
z_torch = neurone(torch.tensor([2.0, -1.0, 0.5]))
print("z (PyTorch) :", round(z_torch.item(), 4), "| a = ReLU(z) :", round(torch.relu(z_torch).item(), 4))

## 2. Les fonctions d'activation

| Activation | Formule | Usage principal |
| --- | --- | --- |
| ReLU | $\max(0, z)$ | Couches cachées (choix par défaut) |
| Sigmoïde | $1 / (1 + e^{-z})$ | Probabilité d'une classe binaire |
| Tanh | $(e^{z} - e^{-z}) / (e^{z} + e^{-z})$ | Sorties centrées entre −1 et 1 |
| Softmax | $e^{z_k} / \sum_j e^{z_j}$ | Probabilités de plusieurs classes |

In [ ]:
z = torch.linspace(-4, 4, 200)
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for ax, (nom, f) in zip(axes, [("ReLU", torch.relu), ("Sigmoïde", torch.sigmoid), ("Tanh", torch.tanh)]):
    ax.plot(z, f(z), lw=2.5); ax.axhline(0, color="gray", lw=0.5); ax.axvline(0, color="gray", lw=0.5); ax.set_title(nom)
plt.tight_layout(); plt.show()

**Pourquoi la non-linéarité est indispensable.** Une composition de fonctions affines est encore une fonction affine : $W_2(W_1 x + b_1) + b_2 = (W_2 W_1)\,x + (W_2 b_1 + b_2)$. Sans activation non linéaire, un réseau de 100 couches équivaut à une seule couche linéaire.

**Prédiction.** Deux couches linéaires sans activation peuvent-elles produire autre chose qu'une droite ?

In [ ]:
torch.manual_seed(1)
sans_activation = nn.Sequential(nn.Linear(1, 8), nn.Linear(8, 1))
avec_relu = nn.Sequential(nn.Linear(1, 8), nn.ReLU(), nn.Linear(8, 1))
xs = torch.linspace(-3, 3, 200).unsqueeze(1)
with torch.no_grad():
    plt.figure(figsize=(6, 3.5))
    plt.plot(xs, sans_activation(xs), lw=2, label="2 couches, sans activation")
    plt.plot(xs, avec_relu(xs), lw=2, label="2 couches, avec ReLU")
plt.title("Sans activation, le réseau reste linéaire"); plt.legend(); plt.show()

## 3. Des neurones aux couches, des couches aux réseaux

Une **couche** applique plusieurs neurones en parallèle aux mêmes entrées. Sous forme matricielle, pour une couche de $m$ neurones et un lot de $n$ observations :

$$Z = X\,W^\top + b \qquad X \in \mathbb{R}^{n \times d},\; W \in \mathbb{R}^{m \times d},\; Z \in \mathbb{R}^{n \times m}$$

Un **réseau multicouche** enchaîne des couches : chaque couche transforme la représentation produite par la précédente. Les couches intermédiaires sont dites **cachées**.

```
Entrées (d)  ->  Couche cachée 1 (16)  ->  ReLU  ->  Couche cachée 2 (16)  ->  ReLU  ->  Sortie (k)
```

Le **théorème d'approximation universelle** établit qu'un réseau à une couche cachée suffisamment large peut approcher toute fonction continue sur un domaine borné. En pratique, les réseaux **profonds** (plusieurs couches) obtiennent les mêmes capacités avec beaucoup moins de paramètres, en construisant des représentations hiérarchiques.

## 4. Les tenseurs PyTorch

Un **tenseur** est un tableau multidimensionnel, analogue à un tableau NumPy, avec deux capacités supplémentaires :
1. il peut résider sur un **GPU** (`.to("cuda")`) ;
2. il peut **enregistrer les opérations** effectuées afin de calculer des gradients (`requires_grad=True`).

| Dimension | Exemple | Forme |
| --- | --- | --- |
| 0 | Valeur de la perte | `()` |
| 1 | Les 4 mesures d'une fleur | `(4,)` |
| 2 | Un lot de 32 fleurs | `(32, 4)` |
| 4 | Un lot de 64 images en niveaux de gris de 28 × 28 | `(64, 1, 28, 28)` : (lot, canaux, hauteur, largeur) |

In [ ]:
appareil = "cuda" if torch.cuda.is_available() else "cpu"
t = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
print("Tenseur :\n", t)
print("Forme :", tuple(t.shape), "| type :", t.dtype, "| appareil :", t.device)
print("Conversion depuis NumPy :", torch.from_numpy(np.arange(3)))
print("Produit matriciel t @ t :\n", t @ t)
print("Déplacement vers", appareil, ":", t.to(appareil).device)

## 5. Autograd et rétropropagation

En IA101, le gradient de l'EQM a été dérivé à la main. Pour un réseau de millions de paramètres, ce calcul est automatisé par **autograd** : PyTorch construit le **graphe de calcul** lors de la passe avant, puis applique la règle de dérivation des fonctions composées en parcourant ce graphe **de la sortie vers les entrées**. C'est la **rétropropagation**.

### Exemple détaillé

Un réseau minimal à deux neurones en chaîne : $h = \text{ReLU}(w_1 x)$, $\hat{y} = w_2 h$, perte $L = (\hat{y} - y)^2$.

Avec $x = 2$, $y = 3$, $w_1 = 0{,}5$, $w_2 = 2$ :
- passe avant : $h = \text{ReLU}(1) = 1$, $\hat{y} = 2$, $L = 1$ ;
- passe arrière, de la sortie vers l'entrée :

$$\frac{\partial L}{\partial \hat{y}} = 2(\hat{y} - y) = -2 \qquad \frac{\partial L}{\partial w_2} = \frac{\partial L}{\partial \hat{y}}\cdot h = -2 \qquad \frac{\partial L}{\partial h} = \frac{\partial L}{\partial \hat{y}}\cdot w_2 = -4$$

$$\frac{\partial L}{\partial w_1} = \frac{\partial L}{\partial h}\cdot \text{ReLU}'(w_1 x)\cdot x = -4 \times 1 \times 2 = -8$$

Chaque gradient se calcule à partir du gradient de la couche suivante : c'est ce qui rend la rétropropagation efficace.

### Exercice 2 — Vérifier la rétropropagation avec autograd

In [ ]:
x, y = torch.tensor(2.0), torch.tensor(3.0)
w1 = torch.tensor(0.5, requires_grad=True)
w2 = torch.tensor(2.0, requires_grad=True)

h = torch.relu(w1 * x)
y_chapeau = w2 * h
perte = (y_chapeau - y) ** 2
...                          # À COMPLÉTER : déclencher la rétropropagation

print(f"perte = {perte.item()}")
print(f"dL/dw2 = {w2.grad.item()}  (calcul à la main : -2)")
print(f"dL/dw1 = {w1.grad.item()}  (calcul à la main : -8)")

<details>
<summary><b>Afficher la solution</b> (après avoir fait l'exercice)</summary>

```python
x, y = torch.tensor(2.0), torch.tensor(3.0)
w1 = torch.tensor(0.5, requires_grad=True)
w2 = torch.tensor(2.0, requires_grad=True)

h = torch.relu(w1 * x)
y_chapeau = w2 * h
perte = (y_chapeau - y) ** 2
perte.backward()

print(f"perte = {perte.item()}")
print(f"dL/dw2 = {w2.grad.item()}  (calcul à la main : -2)")
print(f"dL/dw1 = {w1.grad.item()}  (calcul à la main : -8)")
```

</details>

```
Passe avant  :   x ──> [w1·x] ──> ReLU ──> h ──> [w2·h] ──> ŷ ──> L
Passe arrière:   ∂L/∂w1 <──────────────── ∂L/∂h <──────── ∂L/∂ŷ <── 1
```

`perte.backward()` remplit l'attribut `.grad` de chaque paramètre. La **mise à jour** des paramètres est une étape distincte, réalisée par un optimiseur (notebook 2).

## 6. Définir un réseau

### Option A — `nn.Sequential` : une pile de couches

In [ ]:
modele = nn.Sequential(
    nn.Linear(4, 16),   # 4 variables d'entrée -> 16 neurones
    nn.ReLU(),
    nn.Linear(16, 3),   # 16 -> 3 sorties (une par classe)
)
print(modele)

### Exercice 3 — Formes des tenseurs et nombre de paramètres

Un lot de 32 observations traverse ce réseau.

1. Quelle est la forme du tenseur après chaque couche ?
2. Combien de paramètres le réseau contient-il ? Une couche `Linear(n, m)` possède $n \times m$ poids et $m$ biais.

In [ ]:
formes_prevues = [(32, ...), (32, ...), (32, ...)]   # À COMPLÉTER
nb_parametres_prevu = ...                            # À COMPLÉTER

lot = torch.randn(32, 4)
for couche in modele:
    lot = couche(lot)
    print(f"{couche.__class__.__name__:<8} -> {tuple(lot.shape)}")
print("Nombre de paramètres :", sum(p.numel() for p in modele.parameters()))

<details>
<summary><b>Afficher la solution</b> (après avoir fait l'exercice)</summary>

```python
formes_prevues = [(32, 16), (32, 16), (32, 3)]
nb_parametres_prevu = (4 * 16 + 16) + (16 * 3 + 3)   # 80 + 51 = 131

lot = torch.randn(32, 4)
for couche in modele:
    lot = couche(lot)
    print(f"{couche.__class__.__name__:<8} -> {tuple(lot.shape)}")
print("Nombre de paramètres :", sum(p.numel() for p in modele.parameters()), "| prévu :", nb_parametres_prevu)
```

</details>

### Option B — Sous-classer `nn.Module` (recommandé)

La classe déclare ses couches dans `__init__` et décrit le calcul dans `forward`. Cette forme permet tout calcul (branches, sorties multiples, connexions résiduelles) et constitue la norme dans les projets.

In [ ]:
class ClassifieurMulticouche(nn.Module):
    def __init__(self, n_entrees=4, n_caches=16, n_classes=3):
        super().__init__()
        self.couche1 = nn.Linear(n_entrees, n_caches)
        self.couche2 = nn.Linear(n_caches, n_classes)

    def forward(self, x):
        h = torch.relu(self.couche1(x))
        return self.couche2(h)

modele_b = ClassifieurMulticouche()
print(modele_b)
print("Sortie pour 5 observations :", tuple(modele_b(torch.randn(5, 4)).shape))

## 7. Couche de sortie et fonction de perte

| Tâche | Exemple | Couche de sortie | Fonction de perte |
| --- | --- | --- | --- |
| Régression | Délai d'arrivée | `Linear(…, 1)` | `nn.MSELoss()` |
| Classification binaire | Malin ou bénin | `Linear(…, 1)` | `nn.BCEWithLogitsLoss()` |
| Classification multiclasse | 10 catégories de vêtements | `Linear(…, 10)` | `nn.CrossEntropyLoss()` |

Avec `BCEWithLogitsLoss` et `CrossEntropyLoss`, la sigmoïde ou le softmax sont intégrés à la fonction de perte, pour des raisons de stabilité numérique : le réseau renvoie des scores bruts (*logits*).

In [ ]:
quiz("Que se passe-t-il si l'on empile dix couches Linear sans fonction d'activation ?",
     ["Le réseau devient plus expressif", "Le réseau reste équivalent à une seule couche linéaire", "L'entraînement devient impossible"],
     1, "Une composition de fonctions affines est affine : seules les activations apportent la non-linéarité.")

In [ ]:
quiz("Que fait perte.backward() ?",
     ["Il calcule la prédiction", "Il calcule le gradient de la perte par rapport à chaque paramètre", "Il met à jour les paramètres"],
     1, "backward() remplit les attributs .grad ; la mise à jour est réalisée ensuite par l'optimiseur.")

## À retenir

- **Neurone** : $a = \varphi(w^\top x + b)$ ; une **couche** applique plusieurs neurones en parallèle ; un **réseau** enchaîne des couches.
- Les **activations non linéaires** (ReLU) donnent au réseau sa capacité d'approximation.
- **Tenseurs** : tableaux qui peuvent résider sur GPU et enregistrer les opérations.
- **Rétropropagation** : application de la règle des fonctions composées de la sortie vers l'entrée ; automatisée par `backward()`.
- Définir un réseau : `nn.Sequential` ou une sous-classe de `nn.Module` (`__init__` et `forward`).

**Notebook suivant :** 2 — Entraîner un réseau avec PyTorch.